#  Query Writing: SELECT & Expressions — Full Reference
### With Use-Case Explanations, Syntax, Examples, and Expected Output

This notebook covers the `SELECT`/expression fundamentals from My course, using the pattern:

> **What it's for (the real-world problem it solves) → Syntax → How it's used → Examples with output → Key notes/gotchas**

Examples that use `authors` and `titles` reference the classic SQL Server sample database **`pubs`**, so the sample output reflects real values from that dataset.

In [ ]:
-- OPTIONAL SETUP — only needed if you're running this against a live SQL Server / pubs database
USE pubs;
GO

## 1. `SELECT` — The Foundation of Every Query

### What it's for
`SELECT` is how you tell SQL Server "this is what I want to see in the result." It's the very first thing you write in almost any SQL statement, and — importantly — it doesn't even require a table. You can use it as a quick calculator, a way to test an expression, or a way to preview what a function returns before using it inside a bigger query.

### Syntax
```sql
SELECT expression;
SELECT expression AS column_name;
```

### How it's used
When you're not pulling from a table, `SELECT` just evaluates whatever expression you give it — a literal string, a number, a calculation, a function call — and returns it as a single-row result. This is extremely useful during development: before you plug a function like `DATEDIFF()` or `CONCAT()` into a large query touching a real table, you test it standalone first with a bare `SELECT` to make sure it behaves the way you expect.

In [ ]:
SELECT 'Vahid';

**Output:**

| (No column name) |
|---|
| Vahid |

In [ ]:
SELECT 'Vahid', 'Ghorbani';

**Output:**

| (No column name) | (No column name) |
|---|---|
| Vahid | Ghorbani |

In [ ]:
SELECT 15 * 4;

**Output:**

| (No column name) |
|---|
| 60 |

In [ ]:
SELECT 15 * 4 AS Total;

**Output:**

| Total |
|---|
| 60 |

In [ ]:
SELECT GETDATE();

**Output (example — value depends on when you run it):**

| (No column name) |
|---|
| 2026-09-26 14:32:07.123 |

### Key notes
- Without an alias, SQL Server Management Studio labels the column `(No column name)` — this alone is a strong reason to always alias your expressions.
- Testing an expression with a bare `SELECT` (no `FROM`) costs nothing and catches mistakes early — make it a habit before embedding a function in a complex query.

## 2. Column Aliases with `AS`

### What it's for
An alias renames the output column so the result is meaningful to whoever reads it — a report, a dashboard, an API consumer, or just you six months from now. Without it, calculated columns and expressions show up as generic, unlabeled output, which is unusable outside of quick testing.

### Syntax
```sql
SELECT expression AS alias_name;
```

### How it's used
You attach `AS` right after the expression, followed by the name you want the column to have in the result set. If the name you want contains a space or a reserved word, you must wrap it in square brackets `[Like This]` (or, less ideally, single quotes).

There are two other syntaxes SQL Server accepts — implicit aliasing (`expression alias_name`, no `AS`) and reversed assignment (`alias_name = expression`) — both are valid but considered less explicit and are avoided in clean SQL.

In [ ]:
SELECT 'Vahid' AS FirstName;

**Output:**

| FirstName |
|---|
| Vahid |

In [ ]:
-- Implicit alias (works, but avoid this style)
SELECT 'Vahid' FirstName;

**Output:**

| FirstName |
|---|
| Vahid |

In [ ]:
-- Reversed assignment style (also valid, also avoid in clean code)
SELECT FirstName = 'Vahid';

**Output:**

| FirstName |
|---|
| Vahid |

In [ ]:
-- Alias containing a space, using brackets
SELECT 'Vahid Ghorbani' AS [Full Name];

**Output:**

| Full Name |
|---|
| Vahid Ghorbani |

In [ ]:
-- Alias containing a space, using quotes (also valid, but brackets are more idiomatic in T-SQL)
SELECT 'Vahid Ghorbani' AS 'Full Name';

**Output:**

| Full Name |
|---|
| Vahid Ghorbani |

### Key notes
- Always use `AS` explicitly. *"Explicit is better than implicit"* — it removes any ambiguity about what's an alias versus what's part of the expression.
- Prefer `[Brackets]` over `'Quotes'` for identifiers with spaces — it's the standard T-SQL convention and avoids confusing an identifier with a string literal.

## 3. String Concatenation with `+`

### What it's for
Real-world data is often split across multiple columns — first name and last name, city and zip code, etc. — but a report or UI often needs them combined into a single readable field. The `+` operator is the most basic way to join (concatenate) strings together in T-SQL.

### Syntax
```sql
SELECT value1 + value2 AS alias;
```

### How it's used
When both operands of `+` are strings (or string-compatible types), SQL Server concatenates them end-to-end rather than adding them numerically. To insert a space or separator between values, you explicitly add a string literal space (`' '`) or use the `SPACE()` function between the parts you're joining.

In [ ]:
SELECT 'Vahid' + 'Ghorbani' AS FullName;

**Output:**

| FullName |
|---|
| VahidGhorbani |

In [ ]:
SELECT 'Vahid' + ' ' + 'Ghorbani' + ' ' + 'Sematec' AS FullInfo;

**Output:**

| FullInfo |
|---|
| Vahid Ghorbani Sematec |

### `SPACE(n)` — a companion function
Rather than typing out a literal string of spaces (which is easy to miscount and hard to read), `SPACE(n)` generates exactly `n` space characters. It's self-documenting: anyone reading `SPACE(10)` instantly understands the intent, whereas `'          '` requires counting characters to verify.

In [ ]:
SELECT 'Vahid' + SPACE(10) + 'Ghorbani' AS FullName;

**Output:**

| FullName |
|---|
| Vahid          Ghorbani |

*(Exactly 10 space characters between the two names)*

### Key notes
- `+` concatenation is the most basic approach, but it has a serious weakness covered in the very next section: it breaks completely when any value is `NULL`.
- For joining more than 2–3 fields, `CONCAT_WS()` (Section 6) is almost always a better choice.

## 4. Understanding `NULL`

### What it's for
`NULL` represents **the absence of a value** — not zero, not an empty string, but genuinely "unknown" or "not applicable." You need to understand `NULL` deeply because it behaves differently from every other value in SQL, and it silently breaks a lot of naive query logic if you're not careful.

### Syntax
```sql
SELECT NULL;
```

### How it's used
`NULL` shows up constantly in real data — an author who hasn't signed a contract yet, a book that hasn't been priced, a customer who never provided a phone number. The single most important behavior to memorize: **any arithmetic or string concatenation involving `NULL` using `+` produces `NULL` as the entire result** — not just for that one value, but for the whole expression.

In [ ]:
SELECT NULL;

**Output:**

| (No column name) |
|---|
| NULL |

In [ ]:
SELECT 'Vahid' + NULL + 'Ghorbani' AS FullInfo;

**Output:**

| FullInfo |
|---|
| NULL |

Notice that even though `'Vahid'` and `'Ghorbani'` are perfectly valid strings, the presence of `NULL` in the middle collapses the *entire* concatenated result to `NULL`.

In [ ]:
SELECT 'Vahid' + NULL + 'Ghorbani' + ' ' + 'Sematec' AS FullInfo;

**Output:**

| FullInfo |
|---|
| NULL |

Adding more strings doesn't help — as long as one `NULL` is anywhere in a `+` chain, the whole thing is `NULL`.

### Key notes
- This is the single most common beginner mistake in SQL: building a "full name" or "full address" field with `+` and getting unexpected `NULL` results because one input column happened to be empty.
- The fix is either `ISNULL()` (next section) or switching to `CONCAT`/`CONCAT_WS` (Section 6), which are `NULL`-safe by design.

## 5. `ISNULL()` — Replacing NULL with a Fallback Value

### What it's for
Now that you know `NULL` can silently wreck a concatenation or a calculation, `ISNULL()` is the tool you use to **defend against it**. Its job is simple: "if this value is `NULL`, use this other value instead." You'll use this constantly whenever a column *might* be empty but you need a query to always show something sensible — a `0` instead of a missing price, a placeholder like `'N/A'` instead of a missing phone number, or a space instead of a missing middle name.

### Syntax
```sql
ISNULL(value, replacement)
```

### How it's used
`ISNULL` takes exactly two arguments. It checks the first one — if it's `NULL`, it returns the second argument (your fallback); if it's *not* `NULL`, it returns the original value unchanged. You typically wrap it around any column that's nullable, right before you use that column in a concatenation, a calculation, or a report where a blank/missing value would look broken.

In [ ]:
SELECT ISNULL(NULL, 'Unknown');

**Output:**

| (No column name) |
|---|
| Unknown |

In [ ]:
-- Fixing the broken concatenation from Section 4
SELECT 'Vahid' + ISNULL(NULL, ' ') + 'Ghorbani' AS FullInfo;

**Output:**

| FullInfo |
|---|
| Vahid Ghorbani |

The `NULL` is replaced with a space *before* the `+` concatenation runs, so the whole expression survives instead of collapsing.

In [ ]:
-- Real-world example: a nullable price column
SELECT ISNULL(price, 0) AS Price
FROM dbo.titles
WHERE title_id = 'MC3026';

**Output:**

| Price |
|---|
| 0.00 |

*(This particular book genuinely has a `NULL` price in the `pubs` sample data, so `ISNULL` substitutes `0` instead of showing a blank.)*

In [ ]:
-- Compare with a title that DOES have a price (ISNULL passes it through unchanged)
SELECT ISNULL(price, 0) AS Price
FROM dbo.titles
WHERE title_id = 'BU1032';

**Output:**

| Price |
|---|
| 19.99 |

### Key notes
- `ISNULL` takes **exactly two arguments** — for a chain of possible fallbacks (try A, then B, then C), you'd need to nest `ISNULL` calls or use `COALESCE()` instead, which accepts any number of arguments.
- The replacement value's data type should generally match (or be compatible with) the original column's type — this is why later on we needed `CAST()` to mix a numeric column with a text fallback like `'NA'`.

## 6. `CONCAT()` and `CONCAT_WS()` — Safer String Joining

### What it's for
Both functions solve the exact problem you saw in Section 4: joining multiple strings together **without** the risk of the whole result collapsing to `NULL`. `CONCAT_WS` additionally solves a second, more subtle problem — cleanly inserting a separator (like a space or dash) between multiple values without worrying about double separators when a field happens to be missing.

### Syntax
```sql
CONCAT(value1, value2, value3, ...)
CONCAT_WS(separator, value1, value2, value3, ...)
```
(`WS` stands for **With Separator**.)

### How it's used
Use `CONCAT()` when you just need to glue several values together and don't need a repeated separator between all of them (or you're manually placing separators as separate arguments). Use `CONCAT_WS()` when you're joining a *list* of similar values (name parts, address parts) and want the same separator applied consistently between each one — this is almost always the right tool once you're joining three or more fields.

In [ ]:
-- CONCAT tolerates NULL values without breaking
SELECT CONCAT('Vahid', ' ', 'Ghorbani', NULL, ' ', 'Sematec') AS FullInfo;

**Output:**

| FullInfo |
|---|
| Vahid Ghorbani Sematec |

Unlike `+`, the `NULL` in the middle is simply treated as an empty string and skipped — it does **not** poison the whole result.

In [ ]:
SELECT CONCAT_WS(' ', 'Vahid', 'Ghorbani') AS FullName;

**Output:**

| FullName |
|---|
| Vahid Ghorbani |

In [ ]:
-- Real-world example: joining first and last name from the authors table
SELECT CONCAT_WS(' ', a.au_fname, a.au_lname) AS FullName
FROM dbo.authors AS a
WHERE a.au_id = '213-46-8915';

**Output:**

| FullName |
|---|
| Marjorie Green |

In [ ]:
SELECT a.au_fname
FROM dbo.authors AS a;

In [ ]:
FROM dbo.authors AS a
             ↓
       authors = a

In [ ]:
a.au_fname
│   │
│   └── column
└────── alias of table

In [ ]:
SELECT CONCAT_WS(' ', a.au_fname, a.au_lname) AS FullName
FROM dbo.authors AS a;

In [ ]:
-- Building a full address from four separate columns, dash-separated
SELECT CONCAT_WS('-', a.[state], a.city, a.[address], a.zip) AS Address
FROM dbo.authors AS a
WHERE a.au_id = '213-46-8915';

**Output:**

| Address |
|---|
| CA-Oakland-309 63rd St. #411-94618 |

In [ ]:
-- If every value is NULL, CONCAT_WS returns an empty string, not NULL
SELECT CONCAT_WS(' ', NULL, NULL, NULL, NULL) AS FullInfo;

**Output:**

| FullInfo |
|---|
| *(empty string)* |

### Key notes
- `CONCAT_WS` places the separator **only between actual values** — if one field in the middle is missing, you won't end up with an ugly double-dash or double-space. This is a real advantage over manually chaining `+` and separator literals.
- For joining 3+ fields, default to `CONCAT_WS` over manual `+` chains — it's shorter to write, safer with `NULL`, and easier to read.

## 7. Date Functions: `GETDATE()` and `DATEDIFF()`

### What it's for
Almost every real database has date columns — publish dates, birth dates, order dates, hire dates — and a huge share of business questions are really date-math questions in disguise: *"How old is this record?" "How many days since this order shipped?" "How many years has this employee worked here?"* `GETDATE()` and `DATEDIFF()` together answer exactly this category of question.

### Syntax
```sql
GETDATE()
DATEDIFF(datepart, startdate, enddate)
```

### How it's used
`GETDATE()` simply returns the current date and time from the SQL Server instance — no arguments needed. `DATEDIFF()` takes three arguments: the unit you want the answer in (`YEAR`, `MONTH`, `DAY`, etc.), a start date, and an end date — and returns a single integer representing how many of that unit separate the two dates.

In [ ]:
SELECT GETDATE() AS CurrentDateTime;

**Output (value depends on when you run it):**

| CurrentDateTime |
|---|
| 2026-09-26 14:32:07.123 |

In [ ]:
SELECT DATEDIFF(YEAR, '2002-06-08', GETDATE()) AS Age;

**Output (as of Sept 2026):**

| Age |
|---|
| 24 |

In [ ]:
SELECT DATEDIFF(DAY, '2026-01-01', '2026-09-26') AS DaysPassed;

**Output:**

| DaysPassed |
|---|
| 268 |

In [ ]:
SELECT DATEDIFF(MONTH, '2024-01-01', GETDATE()) AS MonthsPassed;

**Output (as of Sept 2026):**

| MonthsPassed |
|---|
| 32 |

In [ ]:
-- Real-world example: how many years since a book was published
SELECT t.title, DATEDIFF(YEAR, t.pubdate, GETDATE()) AS PublishedYearsAgo
FROM dbo.titles AS t
WHERE t.title_id = 'BU1032';

**Output (as of Sept 2026):**

| title | PublishedYearsAgo |
|---|---|
| The Busy Executive's Database Guide | 41 |

### Key notes
- `DATEDIFF` counts **calendar boundary crossings**, not exact elapsed time. For example, `DATEDIFF(YEAR, '2025-12-31', '2026-01-01')` returns `1`, even though only a single day separates the two dates — because a year boundary (Dec 31 → Jan 1) was crossed. This surprises a lot of beginners.
- Always write the date part in full and uppercase (`YEAR`, `MONTH`, `DAY`) rather than the short forms (`yy`, `mm`, `dd`) — it's more explicit and self-documenting, even though SQL Server accepts both.

## 8. `USE` and `GO`

### What it's for
A single SQL Server instance typically hosts multiple databases at once. `USE` is how you tell your session "point all my unqualified table references at *this* database from now on." `GO` is a client-side signal (used by SSMS, sqlcmd, etc.) that separates your script into distinct execution batches — it's not actually a T-SQL keyword, but a tool-level convention.

### Syntax
```sql
USE database_name;
GO
```

### How it's used
You put `USE database_name;` at the top of a script to switch context, then follow it with `GO` before writing your actual queries. This matters because forgetting which database is "active" is one of the easiest ways to accidentally run a query against the wrong data.

In [ ]:
USE pubs;
GO

SELECT * FROM dbo.titles;

**Output:** all 18 rows from the `titles` table in the `pubs` database.

In [ ]:
-- Switching to a different database later in the same script
USE Northwind;
GO

SELECT * FROM dbo.Customers;

**Output:** rows from the `Customers` table in `Northwind` — note the active database context has now changed.

### Key notes
- `USE` affects the entire session, so if you forget to check which database is active, you risk running a query against the wrong one — always glance at the database dropdown/context at the top of SSMS before executing anything important.
- `GO` batches are also relevant for certain DDL statements (like `CREATE PROCEDURE`) that must be the first statement in their batch — this becomes more important once you start writing stored procedures.

## 9. Fully Qualified Table Names

### What it's for
Sometimes you need to reference a table without relying on whatever database happens to be "active" in your session — for example, when writing a query that joins tables across two different databases on the same server, or when you simply want zero ambiguity about exactly which table you mean.

### Syntax
```
[Database].[Schema].[Table]
```

### How it's used
You chain the database name, schema name, and table name with dots. If you omit the schema (leaving two dots in a row), SQL Server assumes the default schema, `dbo`.

In [ ]:
-- Fully qualified: Database.Schema.Table
SELECT * FROM pubs.dbo.titles;

**Output:** all 18 rows from `titles`, regardless of which database is currently active.

In [ ]:
-- Omitting the schema (defaults to dbo)
SELECT * FROM pubs..titles;

**Output:** identical result to the previous query — the double-dot skips the schema and falls back to `dbo`.

In [ ]:
-- Querying a DIFFERENT database without switching your active context
SELECT * FROM Northwind.dbo.Customers;

**Output:** rows from `Customers` in `Northwind`, even if your session is currently set to `pubs`.

### Key notes
- Fully qualified names become essential once you write cross-database joins, since you can't rely on `USE` to point at two databases simultaneously.
- `dbo` stands for "database owner" and is the default schema in most SQL Server databases — you'll rarely need to change it in a learning environment, but production systems often use multiple schemas to separate concerns (e.g., `Sales.Orders` vs. `HR.Employees`).

## 10. Table Aliases

### What it's for
Once queries start referencing multiple tables (which happens almost immediately once you learn `JOIN`), you need a short, unambiguous way to say "this column comes from that specific table." Table aliases also just make queries shorter and easier to type and read.

### Syntax
```sql
SELECT alias.column
FROM table_name AS alias;
```

### How it's used
Right after naming the table in your `FROM` clause, you add `AS` followed by a short letter or abbreviation (commonly the first letter of the table name, like `a` for `authors` or `t` for `titles`). From that point on in the query, you prefix every column from that table with the alias instead of the full table name.

In [ ]:
-- Without an alias — verbose, and painful once you add a second table
SELECT dbo.authors.au_fname
FROM dbo.authors;

**Output:**

| au_fname |
|---|
| (all first names in the table) |

In [ ]:
-- With an alias — shorter and scales well to multi-table queries
SELECT a.au_fname, a.au_lname
FROM dbo.authors AS a
WHERE a.au_id = '238-95-7766';

**Output:**

| au_fname | au_lname |
|---|---|
| Cheryl | Carson |

In [ ]:
-- Clean multi-line formatting with aliased columns too
SELECT
    a.au_fname AS FirstName,
    a.au_lname AS LastName
FROM
    dbo.authors AS a
WHERE a.au_id = '238-95-7766';

**Output:**

| FirstName | LastName |
|---|---|
| Cheryl | Carson |

### Key notes
- Once you alias a table, be consistent — don't mix `a.au_fname` with the unqualified `au_lname` in the same query; always use the alias for every column.
- Short, meaningful aliases (`a` for authors, `t` for titles) are the convention — avoid overly cryptic single letters once you have 3+ tables in a query (a `JOIN`-heavy query might use `au` and `ti` instead, for clarity).

## 11. Unicode Strings and the `N` Prefix

### What it's for
If you work with non-English text — Persian, Arabic, Chinese, or any script outside the basic ASCII range — you need to tell SQL Server explicitly that a string literal is Unicode. Without this, the text can be silently corrupted (often turning into question marks `????`) when stored or compared.

### Syntax
```sql
N'text'
```

### How it's used
You simply prefix any Unicode string literal with a capital `N` before the opening quote. This applies to literal strings you type directly into a query — it does *not* apply to column or table names, and it's unrelated to the column's actual data type (though the column should ideally be `NVARCHAR`/`NCHAR` to store Unicode properly in the first place).

In [ ]:
SELECT N'وحید قربانی' AS FullName;

**Output:**

| FullName |
|---|
| وحید قربانی |

In [ ]:
-- Using N for a Persian column alias
SELECT
    a.au_fname AS N'نام',
    a.au_lname AS N'نام خانوادگی'
FROM dbo.authors AS a
WHERE a.au_id = '238-95-7766';

**Output:**

| نام | نام خانوادگی |
|---|---|
| Cheryl | Carson |

### Key notes
- `N` is only meaningful on **string literals** — writing `N` before a column name has no effect and is a syntax error if misapplied.
- If you forget the `N` prefix when inserting non-English text into a real `NVARCHAR` column (not just displaying it in a `SELECT`), the data itself can get corrupted on write — this is a much more serious version of the same mistake, so build the habit early.

## 12. `CAST()` — Converting Between Data Types

### What it's for
SQL Server is strict about data types — you generally can't mix a number and a string in the same expression without an explicit conversion. `CAST()` is how you bridge that gap: turning a number into text so it can be concatenated, turning a `datetime` into a plain `date` to drop the time portion, or preparing a numeric column so it can share space with a text fallback (like the `'NA'` pattern from Section 5).

### Syntax
```sql
CAST(value AS datatype)
```

### How it's used
You wrap the value you want to convert with `CAST(... AS target_type)`, where `target_type` includes any size specification it needs, like `VARCHAR(10)` or `NVARCHAR(20)`. This is especially common right before an `ISNULL()` call when you want a numeric column's fallback value to be a text label instead of another number.

In [ ]:
SELECT CAST(1024 AS VARCHAR(10)) AS OrderCode;

**Output:**

| OrderCode |
|---|
| 1024 |

*(This looks identical to the number, but it's now a string — you can concatenate it with `+` like any other text.)*

In [ ]:
SELECT 'Order #' + CAST(1024 AS VARCHAR(10)) AS OrderLabel;

**Output:**

| OrderLabel |
|---|
| Order #1024 |

In [ ]:
-- Combining CAST with ISNULL: a numeric column with a text fallback
SELECT ISNULL(CAST(t.price AS VARCHAR(10)), 'NA') AS Price
FROM dbo.titles AS t
WHERE t.title_id = 'MC3026';

**Output:**

| Price |
|---|
| NA |

In [ ]:
SELECT CAST(GETDATE() AS DATE) AS TodayOnly;

**Output:**

| TodayOnly |
|---|
| 2026-09-26 |

*(The time portion is completely dropped — only the date remains.)*

### Key notes
- Without `CAST`, trying `ISNULL(t.price, 'NA')` directly throws a type-conversion error, because `price` is `money` and `'NA'` is text — SQL Server won't implicitly guess which type should win.
- Pick a reasonably sized target — `VARCHAR(10)` for a short price string, `NVARCHAR(20)` for a Persian label — rather than defaulting to `VARCHAR(MAX)` out of habit; oversized types waste memory in large result sets.

## 13. `CASE WHEN ... THEN ... ELSE ... END` (Searched CASE)

### What it's for
This is SQL's equivalent of if/else-if/else logic in a general-purpose programming language. You use it whenever you need to translate raw data into a more meaningful label, category, or bucket — turning a boolean flag into readable text, translating a status code, or grouping numeric values into ranges (like "low / medium / high" price bands).

### Syntax
```sql
CASE
    WHEN condition THEN result
    WHEN condition THEN result
    ELSE result
END
```

### How it's used
SQL Server evaluates each `WHEN` condition **top to bottom** and returns the result of the *first* one that's true — it does not continue checking the rest after a match. If none of the conditions match, it falls back to `ELSE` (and if you omit `ELSE`, unmatched rows simply return `NULL`). Because each `WHEN` here can hold a full boolean condition (`>`, `<`, `BETWEEN`, `=`, etc.), this is the more flexible of the two `CASE` forms — use it whenever your conditions aren't simple equality checks on a single column.

In [ ]:
SELECT
    a.au_fname,
    a.[contract],
    CASE WHEN a.[contract] = 0 THEN 'No Contract'
         WHEN a.[contract] = 1 THEN 'Has Contract'
         ELSE 'Unknown'
    END AS ContractStatus
FROM dbo.authors AS a
WHERE a.au_id IN ('341-22-1782', '213-46-8915');

**Output:**

| au_fname | contract | ContractStatus |
|---|---|---|
| Meander | 0 | No Contract |
| Marjorie | 1 | Has Contract |

In [ ]:
-- Bucketing a numeric range into readable price bands, using BETWEEN
SELECT
    t.title,
    t.price,
    CASE WHEN t.price BETWEEN 0 AND 10 THEN 'Low Price'
         WHEN t.price BETWEEN 10 AND 20 THEN 'Medium Price'
         WHEN t.price > 20 THEN 'High Price'
         ELSE 'Unknown'
    END AS PriceCategory
FROM dbo.titles AS t
WHERE t.title_id IN ('PS2091', 'BU1032', 'PS1372', 'MC3026');

**Output:**

| title | price | PriceCategory |
|---|---|---|
| Is Anger the Enemy? | 10.95 | Medium Price |
| The Busy Executive's Database Guide | 19.99 | Medium Price |
| Computer Phobic AND Non-Phobic Individuals | 21.59 | High Price |
| The Psychology of Computer Cooking | NULL | Unknown |

### Key notes
- When the input column is `NULL` (like the last row above), **none** of the `WHEN` conditions match — even `BETWEEN` — so the row falls through to `ELSE`. This is a very common source of confusion for beginners who expect `NULL` to somehow match a range check.
- Always include an `ELSE` branch explicitly, even if you think every case is covered — it protects you from silent `NULL` results when new, unexpected data shows up later.
- Indent every `WHEN` at the same level, one line below `CASE`, with `ELSE` and `END` aligned back to the `CASE` keyword — this is the standard formatting convention and makes multi-branch logic much easier to scan.

## 14. Simple `CASE` — Direct Equality Matching

### What it's for
This is a shorthand version of `CASE` for the specific (and very common) situation where every single branch is checking the *same* column or expression for equality against a fixed value — like translating a coded value (`'business'`, `'psychology'`) into a human-readable label. It reads a bit cleaner than the searched form when this is all you need.

### Syntax
```sql
CASE expression
    WHEN value1 THEN result1
    WHEN value2 THEN result2
    ELSE result
END
```

### How it's used
You name the expression being tested once, right after `CASE`, and then each `WHEN` only needs to list the value to compare against — you don't repeat the column name or write `=` explicitly, since equality is implied. The moment you need anything other than equality (a range, a comparison operator, multiple different columns), you must switch back to the searched form from Section 13 — simple `CASE` cannot express those conditions.

In [ ]:
SELECT
    t.title,
    CASE t.[type]
        WHEN 'business' THEN 'Business'
        WHEN 'psychology' THEN 'Psychology'
        WHEN 'mod_cook' THEN 'Modern Cooking'
        ELSE 'Other'
    END AS BookType
FROM dbo.titles AS t
WHERE t.title_id IN ('BU1032', 'PS2091', 'MC2222', 'TC3218');

**Output:**

| title | BookType |
|---|---|
| The Busy Executive's Database Guide | Business |
| Is Anger the Enemy? | Psychology |
| Silicon Valley Gastronomic Treats | Modern Cooking |
| Onions, Leeks, and Garlic: Cooking Secrets of the Mediterranean | Other |

### Key notes
- Simple `CASE` and searched `CASE` produce identical results when your logic is pure equality — the choice is purely about readability, not capability.
- The instant you catch yourself wanting to write `WHEN price > 20` inside a simple `CASE`, that's your signal to switch to the searched form — simple `CASE` syntax has no way to express inequality or range conditions.

## 15. `DISTINCT` — Removing Duplicate Rows

### What it's for
Real tables often have repeated values in a column — many books might share the same `type`, many customers might live in the same city. `DISTINCT` is what you use when you want to know the **unique set of values** present, without caring how many times each one repeats — for example, "what are all the different book categories we carry?" rather than "list every book's category, one row per book."

### Syntax
```sql
SELECT DISTINCT column FROM table_name;
```

### How it's used
You add the keyword `DISTINCT` immediately after `SELECT`. SQL Server then compares full result rows and removes any row that's an exact duplicate of one already shown — this becomes important to remember once you select more than one column at a time.

In [ ]:
SELECT DISTINCT t.[type]
FROM dbo.titles AS t;

**Output:**

| type |
|---|
| business |
| mod_cook |
| popular_comp |
| psychology |
| trad_cook |
| UNDECIDED |

*(Out of 18 total rows in `titles`, only 6 unique type values exist.)*

In [ ]:
-- DISTINCT applied to TWO columns together
SELECT DISTINCT t.[type], t.pub_id
FROM dbo.titles AS t;

**Output (partial example):**

| type | pub_id |
|---|---|
| business | 1389 |
| business | 0736 |
| mod_cook | 0877 |
| psychology | 0736 |
| ... | ... |

Notice that `business` appears **twice** here, once for each different `pub_id` it's paired with — because `DISTINCT` is deduplicating the *combination* of both columns, not each column independently.

### Key notes
- `DISTINCT` applies to the **entire selected row**, not to individual columns in isolation. `SELECT DISTINCT col1, col2` only removes a row if *both* columns match another row exactly.
- `DISTINCT` can be combined with aggregate functions too — for example, `COUNT(DISTINCT Department)` counts how many unique departments exist, which is a preview of the aggregation topics coming up in `GROUP BY`.

# SQL Server Mini Cheat Sheet

## 1. Basic SELECT

```sql
SELECT 'Vahid';

SELECT 'Vahid', 'Ghorbani';

SELECT 15 * 4 AS Total;

SELECT GETDATE();

SELECT NULL;
```

**Idea:** `SELECT` can be used without a table to test expressions, functions, calculations, and values.

---

## 2. Column Aliases

```sql
SELECT 'Vahid' AS FirstName;

SELECT 'Vahid' FirstName;

SELECT FirstName = 'Vahid';

SELECT 'Vahid Ghorbani' AS [Full Name];
```

### Recommended

```sql
SELECT 'Vahid' AS FirstName;
```

Use `AS` explicitly.

For names containing spaces:

```sql
AS [Full Name]
```

---

## 3. String Concatenation with `+`

```sql
SELECT 'Vahid' + ' ' + 'Ghorbani' AS FullName;
```

Using `SPACE()`:

```sql
SELECT 'Vahid' + SPACE(10) + 'Ghorbani' AS FullName;
```

Three values:

```sql
SELECT 'Vahid' + ' ' + 'Ghorbani' + ' ' + 'Sematec' AS FullInfo;
```

---

## 4. NULL and ISNULL()

With `+`, `NULL` can make the entire result `NULL`:

```sql
SELECT 'Vahid' + NULL + 'Ghorbani' AS FullInfo;
```

Use `ISNULL()`:

```sql
SELECT 'Vahid' + ISNULL(NULL, ' ') + 'Ghorbani' AS FullInfo;
```

General syntax:

```sql
ISNULL(expression, replacement_value)
```

Example:

```sql
SELECT ISNULL(price, 0) AS Price
FROM dbo.titles;
```

---

## 5. CONCAT()

General syntax:

```sql
CONCAT(value1, value2, value3, ...)
```

Example:

```sql
SELECT CONCAT('Vahid', ' ', 'Ghorbani') AS FullName;
```

`CONCAT()` handles `NULL` values safely.

---

## 6. CONCAT_WS()

`WS` means **With Separator**.

General syntax:

```sql
CONCAT_WS(separator, value1, value2, ...)
```

Example:

```sql
SELECT CONCAT_WS(' ', 'Vahid', 'Ghorbani') AS FullName;
```

With a table:

```sql
SELECT
    CONCAT_WS(' ', a.au_fname, a.au_lname) AS FullName
FROM dbo.authors AS a;
```

Here:

```text
a              = table alias
a.au_fname     = au_fname column from table a
a.au_lname     = au_lname column from table a
```

Address example:

```sql
SELECT
    CONCAT_WS('-', a.[state], a.city, a.[address], a.zip) AS Address
FROM dbo.authors AS a;
```

---

# 7. Date Functions

Current date and time:

```sql
SELECT GETDATE() AS CurrentDateTime;
```

Difference between dates:

```sql
SELECT DATEDIFF(YEAR, '2002-06-08', GETDATE()) AS Age;
```

Months:

```sql
SELECT DATEDIFF(MONTH, '2024-01-01', GETDATE()) AS MonthsPassed;
```

Days:

```sql
SELECT DATEDIFF(DAY, '2026-01-01', '2026-09-26') AS DaysPassed;
```

General syntax:

```sql
DATEDIFF(datepart, startdate, enddate)
```

---

# 8. USE and Fully Qualified Names

Change database:

```sql
USE pubs;
GO
```

Then:

```sql
SELECT *
FROM dbo.titles;
```

Without changing database:

```sql
SELECT *
FROM pubs.dbo.titles;
```

General object structure:

```text
[Server].[Database].[Schema].[Table]
```

Example:

```sql
SELECT *
FROM Northwind.dbo.Customers;
```

---

# 9. Table Aliases

General syntax:

```sql
FROM table_name AS alias
```

Example:

```sql
FROM dbo.authors AS a
```

This means:

```text
authors → a
```

Therefore:

```sql
a.au_fname
```

means:

```text
au_fname from the authors table
```

Complete example:

```sql
SELECT
    a.au_fname,
    a.au_lname
FROM dbo.authors AS a;
```

Another example:

```sql
SELECT
    t.title_id,
    t.title,
    t.price
FROM dbo.titles AS t;
```

---

# 10. Unicode / Persian Text

For Persian or other Unicode text, use `N`:

```sql
SELECT N'وحید قربانی' AS FullName;
```

Persian alias:

```sql
SELECT
    a.au_fname AS N'نام'
FROM dbo.authors AS a;
```

Multiple Persian aliases:

```sql
SELECT
    a.au_fname AS N'نام',
    a.au_lname AS N'نام خانوادگی'
FROM dbo.authors AS a;
```

---

# 11. CAST()

General syntax:

```sql
CAST(expression AS data_type)
```

Example:

```sql
SELECT CAST(3.875 AS VARCHAR(10)) AS DisplayGPA;
```

Convert date:

```sql
SELECT CAST(GETDATE() AS DATE) AS TodayOnly;
```

Convert number to text:

```sql
SELECT
    'Order #' + CAST(1024 AS VARCHAR(10)) AS OrderLabel;
```

Useful with `ISNULL()`:

```sql
SELECT
    ISNULL(CAST(price AS VARCHAR(10)), 'NA') AS Price
FROM dbo.titles;
```

---

# 12. CASE WHEN

### Searched CASE

Use this when you have conditions such as:

```text
=
>
<
>=
<=
BETWEEN
```

General syntax:

```sql
CASE
    WHEN condition1 THEN result1
    WHEN condition2 THEN result2
    ELSE default_result
END
```

Example:

```sql
SELECT
    t.title,
    t.price,
    CASE
        WHEN t.price BETWEEN 0 AND 10 THEN N'قیمت پایین'
        WHEN t.price BETWEEN 10 AND 20 THEN N'قیمت متوسط'
        WHEN t.price > 20 THEN N'قیمت بالا'
        ELSE N'نامشخص'
    END AS N'دسته بندی قیمت'
FROM dbo.titles AS t;
```

---

# 13. Simple CASE

Use this when comparing the **same expression** for equality.

General syntax:

```sql
CASE expression
    WHEN value1 THEN result1
    WHEN value2 THEN result2
    ELSE default_result
END
```

Example:

```sql
SELECT
    t.title,
    CASE t.[type]
        WHEN 'business' THEN N'کسب و کار'
        WHEN 'psychology' THEN N'روانشناسی'
        ELSE N'نامشخص'
    END AS N'نوع کتاب'
FROM dbo.titles AS t;
```

### Difference

Searched CASE:

```sql
CASE
    WHEN price > 20 THEN 'High'
END
```

Simple CASE:

```sql
CASE type
    WHEN 'business' THEN 'Business'
END
```

---

# 14. DISTINCT

Remove duplicate rows from the result:

```sql
SELECT DISTINCT t.[type]
FROM dbo.titles AS t;
```

Two columns:

```sql
SELECT DISTINCT
    t.[type],
    t.price
FROM dbo.titles AS t;
```

Important:

```sql
DISTINCT
```

applies to the **whole selected row**.

---

# 15. Clean SQL Pattern

For a longer query:

```sql
SELECT
    column1,
    column2,
    expression AS AliasName
FROM dbo.TableName AS t;
```

With conditions:

```sql
SELECT
    t.column1,
    t.column2
FROM dbo.TableName AS t
WHERE condition;
```

For now, focus on understanding:

```text
SELECT
FROM
AS
.
CONCAT
CONCAT_WS
ISNULL
CAST
GETDATE
DATEDIFF
CASE
DISTINCT
```
